# TeNNet-SAC Demonstration Notebook

This notebook demonstrates how to use the TeNNet-SAC framework to:

- Predict **σ-profiles** and molecular geometry from SMILES strings  
- Predict **activity coefficients** for both binary and multicomponent mixtures  
  - You can choose between the **Base model** (trained on COSMO-SAC data) and the **Fine-tuned model** (refined with experimental data)

## Import

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from tennetsac import binary_lng, multi_lng, profile


## Public API

In [ ]:
# Models are initialized lazily on the first prediction call.
# The packaged checkpoints are resolved by tennetsac; no repository paths are needed.

In [ ]:
def plot_sigma_profile(smiles, profile_values):
    sigma = np.linspace(-0.025, 0.025, len(profile_values))
    figure, axis = plt.subplots(figsize=(6, 4))
    axis.plot(sigma, profile_values, linewidth=2)
    axis.set(xlim=(-0.025, 0.025), xlabel="σ (e/Å²)", ylabel="P(σ)", title=f"σ-Profile: {smiles}")
    figure.tight_layout()
    plt.show()
    return figure, axis


def plot_sigma_profile_pair(smiles_1, profile_1, smiles_2, profile_2):
    sigma_1 = np.linspace(-0.025, 0.025, len(profile_1))
    sigma_2 = np.linspace(-0.025, 0.025, len(profile_2))
    figure, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(sigma_1, profile_1, linewidth=2)
    axes[0].set(xlabel="σ (e/Å²)", ylabel="P(σ)", title=f"σ-Profile: {smiles_1}")
    axes[1].plot(sigma_2, profile_2, linewidth=2, color="tab:orange")
    axes[1].set(xlabel="σ (e/Å²)", title=f"σ-Profile: {smiles_2}")
    for axis in axes:
        axis.set_xlim(-0.025, 0.025)
    figure.tight_layout()
    plt.show()
    return figure, axes


def plot_binary_lng(x1_values, temperature, mean_1, mean_2, std_1, std_2, smiles_1, smiles_2):
    x1 = np.asarray(x1_values)
    mean_1, mean_2 = np.asarray(mean_1), np.asarray(mean_2)
    std_1, std_2 = np.asarray(std_1), np.asarray(std_2)
    figure, axis = plt.subplots(figsize=(7, 4.5))
    axis.plot(x1, mean_1, linewidth=2, label="mean ln γ₁")
    axis.plot(x1, mean_2, linewidth=2, label="mean ln γ₂")
    axis.fill_between(x1, mean_1 - std_1, mean_1 + std_1, alpha=0.2, label="±1 std, γ₁")
    axis.fill_between(x1, mean_2 - std_2, mean_2 + std_2, alpha=0.2, label="±1 std, γ₂")
    axis.set(xlim=(0, 1), xlabel="x₁", ylabel="ln γ", title=f"Binary mixture at {temperature:.2f} K\n1: {smiles_1}, 2: {smiles_2}")
    axis.legend()
    figure.tight_layout()
    plt.show()
    return figure, axis


# σ-Profile Prediction

**Input:**  
`smiles`: A single SMILES string

**Output:**  
- Surface area (Å²)  
- Molecular volume (Å³)  
- σ-profile plot

In [ ]:
# ==== Input ====
smiles = r'C/C=C\CCC'
# ================

prf, area, volume = profile(smiles)
print(f"Surface Area    : {area:.2f} Å²")
print(f"Molecular Volume: {volume:.2f} Å³")
plot_sigma_profile(smiles, prf)

# Activity Coefficient Prediction

### Choose Model

Select which model to use for predicting activity coefficients:

1. The **Base model** (trained on COSMO-SAC data)
2. The **Fine-tuned model** (refined with experimental data)

This notebook uses the `"tuned"` ensemble because it provides both the mean and population standard deviation. `"base"` and numbered members require `return_std=False` and return mean-only results.


In [ ]:
version = "tuned"  # Ensemble mean and population standard deviation.

## Binary System

Define the input for binary mixture prediction:

- `smiles_1`: SMILES of component 1  
- `smiles_2`: SMILES of component 2  
- `temperature`: Temperature in Kelvin 
- `x1_list`: Mole fraction list for component 1 (from 0 to 1)

The mole fraction of component 2 will be automatically set as `1 − x₁`.

In [ ]:
# ==== Input ====
smiles_1 = "CCO"
smiles_2 = "ClCCCl"
temperature = 313.15  # K
x1_list = np.linspace(0.0, 1.0, 101)
# ================

ln_gamma_1_mean, ln_gamma_2_mean, ln_gamma_1_std, ln_gamma_2_std = binary_lng(
    [smiles_1, smiles_2], temperature, x1_list.tolist(), version=version
)

plot_binary_lng(
    x1_list, temperature, ln_gamma_1_mean, ln_gamma_2_mean,
    ln_gamma_1_std, ln_gamma_2_std, smiles_1, smiles_2
)

profile_1, _, _ = profile(smiles_1)
profile_2, _, _ = profile(smiles_2)
plot_sigma_profile_pair(smiles_1, profile_1, smiles_2, profile_2)


## Multicomponent System

Define the input for a multicomponent mixture:

- `smiles_list`: List of SMILES strings for all *n* components  
- `mole_fraction_list`: Mole fractions for all *n* components; they must sum to 1  
- `temperature`: Temperature in Kelvin  

The model will predict the activity coefficients (`ln γ`) for each component.


In [ ]:
# ==== Input ====
smiles_list = ['CO', 'CCCc1ccccc1C', r'C/C=C\C', 'O'] # len = n
mole_fraction_list = [0.3, 0.4, 0.1, 0.2] # len = n
temperature = 298.15  # K
# ================

ln_gamma_mean, ln_gamma_std = multi_lng(
    smiles=smiles_list,
    temperature=temperature,
    composition=mole_fraction_list,
    version=version,
)

print("mean ln γ:", ln_gamma_mean)
print("std ln γ:", ln_gamma_std)
